# 💬 1단계 심화 가이드: LangGraph 기반 AI 에이전트 구축 및 고급 스트리밍 아키텍처 (완전 독립실행형)

본 학습용 노트북은 **Paper Agent** 프로젝트의 **대화형 RAG 서비스 (기능 1)** 중 **LangGraph 에이전트 구성과 실시간 스트리밍 통신 기법**을 아주 깊이 있게 이해하기 위해 설계되었습니다.

기존 코드와 달리, 프로젝트 내부의 다른 백엔드 모듈에 의존하지 않고 **이 노트북 단 한 장만으로 모든 데이터베이스 연결, RAG 검색 로직, 툴 정의 및 에이전트 구동이 독립적으로 실행(Self-contained)될 수 있도록** 리팩토링되었습니다. 

학습자는 에이전트가 PostgreSQL pgvector 데이터베이스에서 문서를 유사도 검색하는 로직의 내부 구조를 노트북 상에서 직접 확인하고 실행해 볼 수 있습니다.

---

### 🗺️ LangGraph 에이전트 아키텍처 흐름도

```text
                                    ┌─────────────────────────────┐
                                    │      사용자 질문 (Input)      │
                                    └──────────────┬──────────────┘
                                                   │
                                                   ▼
                                    ┌─────────────────────────────┐
                                    │    LangGraph RAG Agent      │
                                    └──────────────┬──────────────┘
                                                   │
                      ┌────────────────────────────┴────────────────────────────┐
                      ▼                                                         ▼
         [ response_format 활성화 시 ]                            [ response_format 비활성화 시 ]
         (최종 답변의 JSON 구조 보장)                              (실시간 화면 출력 및 인터랙션 보장)
                      │                                                         │
                      ▼                                                         ▼
       - LLM이 답변 전체를 생성한 뒤                           - astream(stream_mode="messages") 활용
         한 번에 파싱하여 결과 반환                              - 실시간 텍스트 토큰 즉각 화면 출력
       - 장점: 백엔드/클라이언트 DTO 규격 준수                 - 도구 호출(Tool Call) 실시간 가집계
       - 단점: 첫 글자가 나올 때까지 대기 시간 김              - 장점: 뛰어난 사용자 경험 (UX)
                      │                                                         │
                      └────────────────────────────┬────────────────────────────┘
                                                   │
                                                   ▼
                                    ┌─────────────────────────────┐
                                    │     AsyncPostgresSaver      │ <─── (체크포인터가 DB에 상태 보존)
                                    └──────────────┬──────────────┘
                                                   │
                                                   ▼
                                    ┌─────────────────────────────┐
                                    │     PostgreSQL database     │ <─── (thread_id 기반 세션 이력 관리)
                                    └─────────────────────────────┘
```

---

### 📌 심화 학습 핵심 포인트

1. **PostgreSQL 기반 체크포인터 (`AsyncPostgresSaver`)**
   - LangGraph에서 멀티 턴(Multi-turn) 이력이 데이터베이스에 자동 누적 및 복원되는 영구 세션 체크포인트 메커니즘을 배웁니다.
2. **구조화 응답(Structured Output) vs 실시간 스트리밍(Streaming)의 딜레마**
   - Pydantic 구조화 출력을 적용할 경우 JSON 스키마 완성 전까지 청크를 전송할 수 없는 한계를 극복하기 위해, **동기식 단답형(`run`)** 에이전트와 **실시간 스트림 전용(`run_stream`)** 에이전트를 이원화하여 설계한 아키텍처 패턴을 이해합니다.
3. **도구 수행 상태와 텍스트 토큰의 이원화 구조**
   - 백엔드 제너레이터가 `type: status`와 `type: token`을 JSON Line 형태로 분할 전송하여 클라이언트가 대기 상태(RAG 검색 등)와 답변 상태를 구분해 화면에 보여주는 규격을 학습합니다.
4. **실시간 스트리밍 파싱 및 출처 가공**
   - 비동기 스트림을 수행하면서 사용된 도구의 상태를 필터링하고 최종 생성된 답변 및 축적된 참고 출처 정보를 조회하는 법을 코드로 직접 확인합니다.


## 1단계. 환경 및 백엔드 설정 로드

노트북 실행 위치를 기반으로 프로젝트 루트의 `.env` 파일을 탐색하고 로드하여 API 키와 데이터베이스 연결 정보를 환경 변수로 주입합니다.

이 노트북은 백엔드 소스 패키지(`api/...`)를 전혀 임포트하지 않고 공식 표준 라이브러리와 외부 패키지(`langchain`, `langgraph`, `psycopg_pool`)만을 직접 사용하여 완전히 자립적으로 동작합니다.

### 💡 주요 핵심 개념
* **`text-embedding-3-large`**: OpenAI가 제공하는 고성능 임베딩 모델로, 본 프로젝트의 데이터 가공 및 RAG 검색을 위해 **3072차원** 규격으로 고정 사용합니다.


In [13]:
import os
import sys
import asyncio
from pathlib import Path
from dotenv import load_dotenv
from typing import Annotated, TypedDict, Any, cast

# 1. 현재 notebooks/chat_service/ 폴더 경로를 기준으로 백엔드(backend) 디렉토리의 절대 경로를 탐색합니다.
current_dir = Path("").resolve() if '__file__' not in locals() else Path(__file__).parent
backend_dir = (current_dir / "../../backend").resolve()

# 2. 백엔드 폴더 내부에 존재하는 .env 파일을 로드하여 환경 변수들을 파이썬 프로세스에 주입합니다.
env_path = backend_dir / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f"✅ 백엔드 환경 설정 로드 완료: {env_path}")
else:
    print(f"⚠️ .env 파일을 찾을 수 없습니다. 경로를 다시 확인해주세요: {env_path}")

# 3. OpenAI API 호출과 데이터베이스 연결을 위한 필수 환경 변수를 메모리에 가져옵니다.
openai_key = os.getenv("OPENAI_API_KEY", "")
database_url = os.getenv("DATABASE_URL", "")

# 4. RAG 및 에이전트 구축에 필요한 LangChain / LangGraph 핵심 라이브러리들을 임포트합니다.
from langchain_openai import OpenAIEmbeddings
from langchain_postgres import PGVector
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import ToolMessage
from langgraph.types import Command
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from langgraph.graph import add_messages
from langchain.agents import create_agent
from pydantic import BaseModel, Field

# 5. text-embedding-3-large 모델 고정 규격 사용 (3072차원)
#    - 백엔드 데이터베이스(pgvector)에 임베딩을 저장하고 검색할 때 차원이 일치해야 하므로 고정해줍니다.
EMBED_MODEL = "text-embedding-3-large"
embeddings_model = OpenAIEmbeddings(
    model=EMBED_MODEL,
    api_key=openai_key
)

# 6. 각 학술 분야(도메인)별로 pgvector 데이터베이스에 매핑되어 있는 컬렉션 이름 정의
#    - RAG 파이프라인에서 분야별로 올바른 테이블(컬렉션)을 바라보게 유도합니다.
DOMAIN_COLLECTIONS = {
    "bio": "bio_embeddings",
    "cs": "cs_embeddings",
    "astronomy": "astronomy_embeddings"
}

print("✅ 기본 모듈 임포트 및 임베딩 모델 설정 완료!")


✅ 백엔드 환경 설정 로드 완료: /Users/pileuszu/Repos/bist-mini-2/backend/.env
✅ 기본 모듈 임포트 및 임베딩 모델 설정 완료!


## 2단계. PostgreSQL 기반 체크포인터 설정 (`AsyncPostgresSaver`)

LangGraph의 대화 기록 영구 저장과 세션 관리를 맡아줄 비동기 PostgreSQL 연결 풀을 직접 생성하고 에이전트 체크포인터와 연동합니다.

### 💡 주요 핵심 개념
1. **개별 DB 연결 풀 인라인 구현**
   - 기존의 백엔드 파일인 `api.database.config.psycopg_pool` 모듈에 의존하지 않고, 주피터 노트북 내부에서 직접 `psycopg_pool.AsyncConnectionPool` 인스턴스를 선언하고 구동합니다.
2. **`database_url` 주소 가공**
   - SQLAlchemy용 프로토콜인 `postgresql+asyncpg://` 형식을 psycopg3 라이브러리가 해석할 수 있는 비동기 커넥션 주소 표준(예: `postgresql://`)으로 문자열을 교체해 인입합니다.


In [14]:
from psycopg_pool import AsyncConnectionPool

# 1. database_url 문자열에서 psycopg 비동기 전용인 postgresql:// 주소 형식으로 변환합니다.
db_conn_string = database_url.replace("postgresql+asyncpg://", "postgresql://")

# 2. AsyncPostgresSaver 대화 기록 체크포인터용 비동기 연결 풀을 직접 선언합니다.
#    - min_size=1, max_size=3 설정으로 불필요한 DB 자원 낭비를 줄입니다.
chat_psycopg_pool = AsyncConnectionPool(
    db_conn_string,
    min_size=1,
    max_size=3,
    kwargs={"autocommit": True},
    open=False,
)

# 3. 비동기 연결 풀을 안전하게 열어줍니다.
if chat_psycopg_pool.closed:
    await chat_psycopg_pool.open()
    print("✅ psycopg 연결 풀 직접 개설 완료!")
else:
    print("ℹ️ psycopg 연결 풀이 이미 열려 있습니다.")

# 4. AsyncPostgresSaver 체크포인터에 psycopg 연결 풀을 주입하여 선언합니다.
checkpointer = AsyncPostgresSaver(cast(Any, chat_psycopg_pool))

# 5. 데이터베이스 내에 checkpoints 테이블이 존재하는지 멱등하게 확인하는 로직입니다.
async with chat_psycopg_pool.connection() as conn:
    cur = await conn.execute(
        "SELECT 1 FROM pg_tables WHERE schemaname='public' AND tablename='checkpoints'"
    )
    exists = await cur.fetchone()
    
    if not exists:
        print("🔧 checkpoints 테이블이 존재하지 않습니다. 마이그레이션 테이블 정리 및 재생성을 진행합니다...")
        await conn.execute("DROP TABLE IF EXISTS checkpoint_migrations, checkpoint_blobs, checkpoint_writes CASCADE;")
        # AsyncPostgresSaver가 대화 이력 관리에 필요한 DB 테이블 및 인덱스를 자동 생성합니다.
        await checkpointer.setup()
        print("✅ checkpoints 테이블 및 관련 마이그레이션 테이블 생성 완료!")
    else:
        print("✅ checkpoints 테이블이 이미 존재하며, 에이전트와 연결할 준비가 되었습니다.")


✅ psycopg 연결 풀 직접 개설 완료!
✅ checkpoints 테이블이 이미 존재하며, 에이전트와 연결할 준비가 되었습니다.


## 3단계. 구조화된 응답을 제공하는 메인 에이전트 생성

학습용 가이드 노트북의 독립성을 완벽히 확보하기 위하여, 기존 백엔드 라이브러리(`api.common.rag_pipeline`, `api.common.tools`)에 내장되어 있던 **RAG 파이프라인(유사도 검색 클래스 및 도메인별 툴)**과 **웹 검색, 시간 툴**의 전체 원천 코드를 이 코드 셀 내부에 인라인화하여 재정의합니다.

이를 통해 RAG 검색이 데이터베이스에 어떤 방식으로 질의(`asimilarity_search_with_score`)를 던지고 문서와 코사인 유사도를 계산하여 결과를 포맷팅하는지 전체 소스 코드를 학습할 수 있습니다.

### 💡 주요 핵심 개념
* **`PGVector` 유사도 검색**:
  - `langchain_postgres`가 제공하는 PGVector를 사용하여 DB 컬렉션에 임베딩 질의를 던지고 유사도 순서로 문서 청크를 추출합니다.
  - 검색된 결과 점수(Distance)를 코사인 유사도 점수(`1.0 - score`)로 가공하고 DTO용 메타데이터(`arxiv_id`, `title`)를 안전하게 추출합니다.
  - **⚠️ `create_extension=False` 방어 코드 설정**:
    - pgvector는 비동기 초기화 과정에서 `create_extension=True` 일 때 `CREATE EXTENSION`과 다른 쿼리를 복수로 묶어 던집니다.
    - 비동기 드라이버인 `asyncpg`는 단일 prepared statement 내 복수 세미콜론 실행을 허용하지 않아 오류(`cannot insert multiple commands into a prepared statement`)를 발생시키므로, 이를 방지하기 위해 반드시 `create_extension=False` 옵션을 명시적으로 걸어주어야 오류가 발생하지 않습니다.
* **`BioAgentState` 내의 `structured_response` 채널 정의**:
  - **중요**: `create_agent`에 `response_format`을 지정하여 구조화 응답을 활용할 때, 반환되는 최종 구조는 상태 스키마(`BioAgentState`)에 정의되어 있어야 합니다.
  - 만약 `BioAgentState` 스키마 내에 `structured_response` 필드가 빠져있다면, LangGraph는 상태 업데이트 시 해당 키를 무시하고 드랍하여 결국 최종 반환 데이터에 포함되지 않아 `KeyError`를 야i기합니다. 따라서 스키마 상에 명시적으로 `structured_response: BioAnswer` 필드를 추가해 줍니다.


In [15]:
import math
from datetime import datetime
from zoneinfo import ZoneInfo

# =====================================================================
# 1. 공통 RAG 검색 파이프라인 (Common RAG Pipeline) 인라인 구현
# =====================================================================
class NotebookCommonRagPipeline:
    """노트북 내부에서 독립적으로 RAG 데이터베이스를 연동하는 유사도 검색기 클래스입니다.
    외부 커스텀 패키지 의존성 없이 PGVector 데이터베이스를 직접 호출합니다.
    """
    def __init__(self, connection_url: str, embeddings) -> None:
        self.connection_url = connection_url
        self.embeddings = embeddings

    async def similarity_search(self, domain: str, query: str, k: int = 3) -> list[dict[str, Any]]:
        # 지원하지 않는 도메인 방어 코드
        if domain not in DOMAIN_COLLECTIONS:
            raise ValueError(f"지원하지 않는 도메인입니다: {domain}")

        collection_name = DOMAIN_COLLECTIONS[domain]
        
        # PGVector 객체를 초기화하여 질의에 대한 유사 문서 탐색을 실행합니다.
        # - 중요: create_extension=False 설정을 통해 asyncpg 드라이버의 prepared statement 다중 명령 제약 예외를 방지합니다.
        vectorstore = PGVector(
            embeddings=self.embeddings,
            collection_name=collection_name,
            connection=self.connection_url,
            async_mode=True,
            create_extension=False,
        )

        # 데이터베이스에 비동기식 유사도 검색 질의를 던져 문장과 거리값(score)을 획득합니다.
        results = await vectorstore.asimilarity_search_with_score(query, k=k)

        def safe_str(val: Any) -> str:
            if val is None:
                return ""
            if isinstance(val, float) and math.isnan(val):
                return ""
            return str(val)

        formatted_results = []
        for doc, score in results:
            meta = doc.metadata or {}
            # pgvector 테이블의 메타데이터 필드 명칭 호환성 파싱
            arxiv_id = meta.get("arxiv_id") or meta.get("doc_id") or ""
            title = meta.get("title", "")

            # 코사인 유사도 점수 가공 (1.0 - 거리값 점수)
            similarity = round(1.0 - score, 4)

            formatted_results.append({
                "doc_id": safe_str(arxiv_id),
                "title": safe_str(title),
                "text_chunk": safe_str(doc.page_content),
                "score": similarity
            })

        return formatted_results

# RAG 파이프라인 서비스 선언 (1단계에서 주입한 데이터베이스 URL 및 임베딩 모델 사용)
common_rag_pipeline = NotebookCommonRagPipeline(database_url, embeddings_model)

# =====================================================================
# 2. 에이전트용 LangChain 도구(Tools) 인라인 정의
# =====================================================================
@tool
async def search_bio_papers(query: str, runtime: ToolRuntime, k: int = 3) -> Command:
    """생명공학·유전체학(q-bio.GN) 논문 데이터베이스에서 관련 내용을 검색합니다.
    유전체학, 유전자 매핑, 시퀀싱 분석 등 생명공학/유전학 질문에 대답하거나 참고 자료가 필요할 때 이 툴을 사용하세요.
    """
    results = await common_rag_pipeline.similarity_search("bio", query, k=k)

    if not results:
        msg = f"q-bio.GN 논문에서 '{query}'와 관련된 내용을 찾을 수 없습니다."
        return Command(update={
            "messages": [ToolMessage(content=msg, tool_call_id=runtime.tool_call_id)],
        })

    output_lines = [f"검색 결과: '{query}' (q-bio.GN 논문)\n", "=" * 80]
    sources = []
    for idx, r in enumerate(results, 1):
        arxiv_id = r["doc_id"]
        title = r["title"]
        score = r["score"]
        output_lines.append(f"\n[논문 {idx}] (유사도: {score:.4f})")
        output_lines.append(f"제목: {title}")
        output_lines.append(f"arXiv ID: {arxiv_id}")
        output_lines.append(f"\n내용:\n{r['text_chunk']}\n")
        output_lines.append("-" * 80)
        
        snippet = " ".join(r.get("text_chunk", "").split())
        if len(snippet) > 160:
            snippet = snippet[:160].rstrip() + "…"
        sources.append({"arxiv_id": arxiv_id, "title": title, "summary": snippet})

    tool_text = "\n".join(output_lines)
    return Command(update={
        "messages": [ToolMessage(content=tool_text, tool_call_id=runtime.tool_call_id)],
        "sources": sources,
    })

@tool
async def search_cs_papers(query: str, runtime: ToolRuntime, k: int = 3) -> Command:
    """컴퓨터 과학(cs.NE) 관련 학술 논문 데이터베이스에서 관련 내용을 검색합니다.
    인공신경망, 진화 알고리즘, 딥러닝 등에 대한 정보가 필요하거나 질문에 대답할 때 이 툴을 사용하세요.
    """
    results = await common_rag_pipeline.similarity_search("cs", query, k=k)

    if not results:
        msg = f"cs.NE 논문에서 '{query}'와 관련된 내용을 찾을 수 없습니다."
        return Command(update={
            "messages": [ToolMessage(content=msg, tool_call_id=runtime.tool_call_id)],
        })

    output_lines = [f"검색 결과: '{query}' (cs.NE 논문)\n", "=" * 80]
    sources = []
    for idx, r in enumerate(results, 1):
        arxiv_id = r["doc_id"]
        title = r["title"]
        score = r["score"]
        output_lines.append(f"\n[논문 {idx}] (유사도: {score:.4f})")
        output_lines.append(f"제목: {title}")
        output_lines.append(f"arXiv ID: {arxiv_id}")
        output_lines.append(f"\n내용:\n{r['text_chunk']}\n")
        output_lines.append("-" * 80)

        snippet = " ".join(r.get("text_chunk", "").split())
        if len(snippet) > 160:
            snippet = snippet[:160].rstrip() + "…"
        sources.append({"arxiv_id": arxiv_id, "title": title, "summary": snippet})

    tool_text = "\n".join(output_lines)
    return Command(update={
        "messages": [ToolMessage(content=tool_text, tool_call_id=runtime.tool_call_id)],
        "sources": sources,
    })

@tool
async def search_astronomy_papers(query: str, runtime: ToolRuntime, k: int = 3) -> Command:
    """지구 및 행성 천체물리학(astro-ph.EP) 논문 데이터베이스에서 관련 내용을 검색합니다.
    행성 대기, Mars 지질학, 행성 형성 등에 관한 질문에 대답하거나 참고 자료가 필요할 때 이 툴을 사용하세요.
    """
    results = await common_rag_pipeline.similarity_search("astronomy", query, k=k)

    if not results:
        msg = f"astro-ph.EP 논문에서 '{query}'와 관련된 내용을 찾을 수 없습니다."
        return Command(update={
            "messages": [ToolMessage(content=msg, tool_call_id=runtime.tool_call_id)],
        })

    output_lines = [f"검색 결과: '{query}' (astro-ph.EP 논문)\n", "=" * 80]
    sources = []
    for idx, r in enumerate(results, 1):
        arxiv_id = r["doc_id"]
        title = r["title"]
        score = r["score"]
        output_lines.append(f"\n[논문 {idx}] (유사도: {score:.4f})")
        output_lines.append(f"제목: {title}")
        output_lines.append(f"arXiv ID: {arxiv_id}")
        output_lines.append(f"\n내용:\n{r['text_chunk']}\n")
        output_lines.append("-" * 80)

        snippet = " ".join(r.get("text_chunk", "").split())
        if len(snippet) > 160:
            snippet = snippet[:160].rstrip() + "…"
        sources.append({"arxiv_id": arxiv_id, "title": title, "summary": snippet})

    tool_text = "\n".join(output_lines)
    return Command(update={
        "messages": [ToolMessage(content=tool_text, tool_call_id=runtime.tool_call_id)],
        "sources": sources,
    })

@tool
async def search_web(query: str, runtime: ToolRuntime, max_results: int = 4) -> Command:
    """논문 데이터베이스(arXiv)에서 적절한 자료를 찾지 못했을 때 웹에서 정보를 검색하는 도구.
    최신 동향, 일반 상식, arXiv 범위 밖 주제 등 논문 검색으로 답할 수 없을 때만 사용하세요.
    """
    try:
        tavily = TavilySearchResults(max_results=max_results, search_depth="basic")
        results = await tavily.ainvoke(query)
    except Exception as e:
        msg = f"웹 검색에 실패했습니다({type(e).__name__}). 웹 정보 없이 답변하거나, 모른다고 안내하세요."
        return Command(update={
            "messages": [ToolMessage(content=msg, tool_call_id=runtime.tool_call_id)],
        })

    if isinstance(results, str):
        return Command(update={
            "messages": [ToolMessage(content=results, tool_call_id=runtime.tool_call_id)],
        })

    if not results:
        msg = f"웹에서 '{query}'에 대한 결과를 찾지 못했습니다."
        return Command(update={
            "messages": [ToolMessage(content=msg, tool_call_id=runtime.tool_call_id)],
        })

    output_lines = [f"웹 검색 결과: '{query}'\n", "=" * 80]
    web_sources = []
    for idx, r in enumerate(results, 1):
        if not isinstance(r, dict):
            continue
        url = r.get("url", "")
        title = r.get("title", "") or url
        content = r.get("content") or ""

        output_lines.append(f"\n[{idx}] {title}")
        output_lines.append(f"URL: {url}")
        output_lines.append(f"내용:\n{content}\n")
        output_lines.append("-" * 80)

        snippet = " ".join(content.split())
        if len(snippet) > 200:
            snippet = snippet[:200].rstrip() + "…"
        web_sources.append({"url": url, "title": title, "summary": snippet})

    tool_text = "\n".join(output_lines)
    return Command(update={
        "messages": [ToolMessage(content=tool_text, tool_call_id=runtime.tool_call_id)],
        "web_sources": web_sources,
    })

@tool
def get_current_datetime() -> str:
    """현재 날짜와 시간을 한국 시간(Asia/Seoul) 기준 ISO-8601 문자열로 반환합니다.
    '오늘', '지금', '현재', '최근' 등 현재 시점이 필요한 질문에 사용하세요.
    """
    return datetime.now(ZoneInfo("Asia/Seoul")).isoformat()

# =====================================================================
# 3. Pydantic DTO 선언 및 에이전트 상태(State Schema) 정의
# =====================================================================
class BioPaperRef(BaseModel):
    arxiv_id: str = Field(description="논문의 arXiv ID (예: 2103.12345)")
    title: str = Field(description="논문 제목")
    summary: str = Field(description="논문의 핵심 내용과 현재 질문과의 구체적인 연관성 요약")

class BioAnswer(BaseModel):
    explanation: str = Field(description="질문에 대한 과학적/기술적 마크다운 설명")
    papers: list[BioPaperRef] = Field(default_factory=list, description="답변의 직접적인 근거가 된 논문들의 목록")

# 에이전트가 내부에 유지할 상태 변수들의 스키마(State Schema)를 정의합니다.
# - KeyError 방지를 위해 구조화 답변 결과 필드인 structured_response를 명시적으로 추가합니다.
class BioAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    sources: list[dict]        # 검색된 논문 출처 누적
    web_sources: list[dict]    # 웹 검색 출처 누적
    structured_response: BioAnswer # 구조화 답변 보관 채널

model_name = "openai:gpt-4o-mini"
system_prompt = """당신은 생명공학·유전체학(q-bio.GN), 천문학(astro-ph.EP), 컴퓨터과학(cs.NE) 논문을 다루는 전문 학술 연구 조력자입니다.
- 중요: 검색 도구(tools)에 전달하는 검색 쿼리(query)는 반드시 영어(English)로 작성해 전달해야 정확한 매칭이 일어납니다.
- 질문의 과학적 주제에 맞춰 가장 알맞은 검색 도구(bio, astronomy, cs 중)를 적절히 호출해 지식을 확장하세요.
- 최종 반환할 papers 리스트에는 답변에 핵심적인 기여를 한 논문 정보들을 명확히 정리해 기재해야 합니다.
- 답변은 항상 사용자가 질문한 한글/영어 언어에 맞춰 친절하게 작성해 주세요."""

main_agent = create_agent(
    model=model_name,
    tools=[search_bio_papers, search_astronomy_papers, search_cs_papers, search_web, get_current_datetime],
    system_prompt=system_prompt,
    checkpointer=checkpointer,
    state_schema=cast(Any, BioAgentState),
    response_format=BioAnswer
)
print("✅ 구조화 출력이 활성화된 메인 에이전트 생성 완료!")


✅ 구조화 출력이 활성화된 메인 에이전트 생성 완료!


## 4단계. 멀티턴(Multi-turn) 대화 및 체크포인트 복원 테스트

앞서 데이터베이스에 연동한 체크포인터가 실제로 정상 구동하는지 동일 세션 키(`thread_id`)를 활용하여 멀티턴 대화 시뮬레이션을 진행합니다.

### 💡 주요 핵심 개념
1. **`thread_id`와 대화 격리**
   - 여러 사용자의 고유 대화 세션을 구분하기 위한 고유 키값입니다. 이 키를 기반으로 DB 내 대화 흐름이 묶이게 됩니다.
2. **컨텍스트 인지 (Context Awareness)**
   - 두 번째 질문("방금 설명해 준 내용 중 가장 중요한 기법...")은 대명사("방금 설명해 준 내용")를 담고 있어, 이전 대화 기록이 유실되면 LLM이 답변할 수 없습니다. DB에서 이전 체크포인트 상태가 정확히 리로드되는지 검증합니다.
3. **⚠️ 동일 세션(thread_id) 재사용 시 주의사항**
   - **중요**: 노트북 테스트 도중 시간 초과나 예외가 발생해 마지막 툴 호출(Tool Call)에 대한 응답(`ToolMessage`)이 DB에 정상 기록되지 않은 채 트랜잭션이 중단될 수 있습니다.
   - 이 상태에서 동일한 고정 `thread_id`를 재사용하여 다시 대화를 실행하면, LangGraph는 DB로부터 불완전하게 깨진 메시지 기록을 복원하여 OpenAI에 그대로 보냅니다.
   - OpenAI는 **"툴 호출 뒤에 툴 응답이 누락되었다"**며 `BadRequestError (400)`를 던지게 됩니다. 
   - 따라서 안전한 테스트를 위해 매 실행 시 `uuid.uuid4()`를 사용하여 고유한 신규 세션을 분리하는 것이 실무적으로 권장됩니다.
4. **`aget_state`**
   - 데이터베이스에서 직접 현재 `thread_id` 세션에 누적되어 적재된 대화 상태 객체(대화 전체 히스토리, 가용한 임베딩 출처 등)를 불러오는 핵심 검증 API입니다.


In [16]:
import uuid

# 1. 동일한 대화방 세션을 지칭할 고유 식별자(thread_id)를 정의합니다.
#    - 중요: 이전 테스트의 깨진 상태 복원으로 인한 400 에러를 방지하기 위해 매 실행마다 uuid로 고유 세션을 격리합니다.
thread_id = f"test-session-{uuid.uuid4().hex[:8]}"
config = {"configurable": {"thread_id": thread_id}}

first_question = "Bioinformatics 분야에서 딥러닝이 어떻게 활용되나요?"
second_question = "방금 설명해 준 내용 중에서 가장 중요한 기법 한 가지만 꼽아서 요약해줘."

# 2. 첫 번째 질문 비동기 전송
print(f"🤖 [1차 질문 전송 | 세션 ID: {thread_id}] " + first_question)
result1 = await main_agent.ainvoke(
    {"messages": [{"role": "user", "content": first_question}], "sources": [], "web_sources": []},
    config
)

print("\n=== [1차 답변 결과 (구조화 DTO 데이터 확인)] ===")
# response_format을 지정했으므로 result1 내부에는 Pydantic 객체인 'structured_response'가 반환됩니다.
structured_response = result1["structured_response"]
print(f"설명(explanation): {structured_response.explanation[:250]}...")
print(f"참고 논문 개수(papers count): {len(structured_response.papers)}")
for idx, paper in enumerate(structured_response.papers, 1):
    print(f"  [{idx}] arXiv ID: {paper.arxiv_id} | 제목: {paper.title[:50]}...")

# 3. 대화 기록이 자동으로 연동 및 유지되는지 확인하기 위해 '맥락적' 두 번째 질문을 전송합니다.
print(f"\n🤖 [2차 맥락 질문 전송] {second_question}")
result2 = await main_agent.ainvoke(
    {"messages": [{"role": "user", "content": second_question}], "sources": [], "web_sources": []},
    config
)

print("\n=== [2차 답변 결과 (컨텍스트 인지 완료)] ===")
structured_response2 = result2["structured_response"]
print(f"설명(explanation): {structured_response2.explanation}")

# 4. 데이터베이스(PostgreSQL)의 세션 스토리지에서 직접 상태를 복원하여 상태 로그를 출력해봅니다.
state = await main_agent.aget_state(config)
print(f"\n✅ DB 체크포인트 저장 상태: 누적된 메시지 개수 = {len(state.values['messages'])}")
for idx, msg in enumerate(state.values['messages']):
    # 메시지 객체의 타입을 알기 쉽게 맵핑합니다.
    role = "User" if msg.type == "human" else "AI" if msg.type == "ai" else msg.type
    print(f"  [{idx}] {role}: {str(msg.content)[:80]}...")


🤖 [1차 질문 전송 | 세션 ID: test-session-c084fe7f] Bioinformatics 분야에서 딥러닝이 어떻게 활용되나요?


Deserializing unregistered type __main__.BioAnswer from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'BioAnswer')]



=== [1차 답변 결과 (구조화 DTO 데이터 확인)] ===
설명(explanation): 딥러닝은 생명정보학 분야에서 대규모 데이터 처리와 패턴 인식 문제를 해결하는 데 매우 효과적입니다. 최근 연구들은 딥러닝 기술이 생물학적 데이터의 패턴을 인식하는 데 어떻게 활용되고 있는지를 보여주고 있습니다....
참고 논문 개수(papers count): 5
  [1] arXiv ID: 1903.00342 | 제목: Deep learning in bioinformatics: introduction, app...
  [2] arXiv ID: 1603.06430 | 제목: Deep Learning in Bioinformatics...
  [3] arXiv ID: 2003.00108 | 제목: Deep Learning in Mining Biological Data...
  [4] arXiv ID: 1802.09791 | 제목: Bioinformatics and Medicine in the Era of Deep Lea...
  [5] arXiv ID: 2406.08686 | 제목: Opportunities in deep learning methods development...

🤖 [2차 맥락 질문 전송] 방금 설명해 준 내용 중에서 가장 중요한 기법 한 가지만 꼽아서 요약해줘.


Deserializing unregistered type __main__.BioAnswer from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'BioAnswer')]



=== [2차 답변 결과 (컨텍스트 인지 완료)] ===
설명(explanation): 딥러닝에서 가장 중요한 기법 중 하나는 **합성곱 신경망(Convolutional Neural Network, CNN)**입니다. CNN은 이미지와 같은 2차원 데이터 분석에 매우 효과적으로, 생물정보학에서 생물학적 이미지 분석, 유전자 발현 패턴 탐지 등에 활용됩니다. CNN은 데이터에서 특징을 자동으로 학습하고 이를 기반으로 분류 및 예측 작업을 수행하여, 다양한 생물학적 문제 해결에 기여하고 있습니다.

✅ DB 체크포인트 저장 상태: 누적된 메시지 개수 = 8
  [0] User: Bioinformatics 분야에서 딥러닝이 어떻게 활용되나요?...
  [1] AI: ...
  [2] tool: 검색 결과: 'deep learning in bioinformatics' (q-bio.GN 논문)

========================...
  [3] AI: {"explanation":"딥러닝은 생명정보학 분야에서 대규모 데이터 처리와 패턴 인식 문제를 해결하는 데 매우 효과적입니다. 최근 연구들은 ...
  [4] User: 방금 설명해 준 내용 중에서 가장 중요한 기법 한 가지만 꼽아서 요약해줘....
  [5] AI: ...
  [6] tool: 검색 결과: 'deep learning architecture bioinformatics' (q-bio.GN 논문)

==============...
  [7] AI: {"explanation":"딥러닝에서 가장 중요한 기법 중 하나는 **합성곱 신경망(Convolutional Neural Network, CN...


## 5단계. 실시간 스트리밍 에이전트 선언 및 청크 가공 시뮬레이션

사용자 인터페이스(UI) 측면에서, AI의 답변이 화면에 실시간으로 흘러나오는 스트리밍 응답은 매우 중요합니다. 하지만 앞서 살펴본 구조화 출력 에이전트로는 스트리밍이 제한됩니다.

### 💡 아키텍처 우회 기법: 이원화 설계
* **스트리밍 전용 에이전트 (`stream_agent`)**:
  - `response_format` 규격을 걸어놓지 않은 상태로 에이전트를 선언합니다. 이 에이전트는 일반 텍스트로 자유롭게 출력을 생성하므로, 실시간 토큰 전송(`astream`)이 막힘 없이 수행됩니다.

### 💡 스트리밍 파싱 로직 흐름도
```text
                      astream(stream_mode="messages")
                                     │
                                     ▼
                      각 비동기 청크 (chunk) 수신
                                     │
             ┌───────────────────────┴───────────────────────┐
             ▼                                               ▼
     [ 툴 호출 이벤트 감지 ]                           [ 텍스트 답변 토큰 감지 ]
   (tool_calls / tool_call_chunks)                    (content 필드에 문자열 존재)
             │                                               │
             ▼                                               ▼
- 사용자에게 "학술 검색 수행 중" 상태 전송          - 답변 텍스트 실시간 출력 (end="", flush=True)
- 중복 이벤트 호출 방지 장치 가동                      - 도구 실행 도중 발생한 내부 AI 토큰 필터링
```

* **`stream_mode="messages"`**:
  - LangGraph의 스트리밍 옵션으로, 대화 노드 및 툴 노드에서 발생하는 메시지의 조각들을 실시간 수집하는 데 최적인 모드입니다.


In [17]:
# 1. 스트리밍 전용 에이전트 선언 (response_format을 제외하여 스트리밍 제약 해제)
stream_system_prompt = """당신은 생명공학·유전체학(q-bio.GN), 천문학(astro-ph.EP), 컴퓨터과학(cs.NE) 논문을 다루는 전문 학술 연구 조력자입니다.
- 중요: 검색 도구(tools)에 전달하는 쿼리는 반드시 영어(English)로 작성해 전달하세요.
- RAG를 거쳐 사용된 논문 정보나 출처는 데이터 백엔드에서 따로 가공되어 프론트엔드로 전달됩니다. 
- 따라서 AI 본문 내에서는 별도로 논문 목록을 번호로 나열하거나, 참고 문헌 섹션을 인위적으로 생성해 지면을 낭비하지 마세요.
- 질문에 대한 자연스러운 과학적 설명 마크다운만 상세하고 풍부하게 작성해 주세요."""

stream_agent = create_agent(
    model=model_name,
    tools=[search_bio_papers, search_astronomy_papers, search_cs_papers, search_web, get_current_datetime],
    system_prompt=stream_system_prompt,
    checkpointer=checkpointer,
    state_schema=cast(Any, BioAgentState)
)
print("✅ 스트리밍 전용 에이전트 초기화 완료!")

# 2. 스트리밍 시뮬레이션 실행 및 데이터 파싱
print("🚀 [Streaming Start] 실시간 스트림 이벤트를 감지하고 토큰을 가공합니다...\n")
question = "천문학 분야에서 planetesimal formation에 대한 논문을 찾아서 설명해줘."

# 툴의 원래 이름(search_web 등)을 클라이언트가 인식할 깔끔한 상태 이벤트 형태의 텍스트로 치환할 맵입니다.
TOOL_STATUS = {
    "search_web": "web_search",
    "search_bio_papers": "paper_search",
    "search_astronomy_papers": "paper_search",
    "search_cs_papers": "paper_search",
    "get_current_datetime": "datetime",
}
announced_tools = set()

# stream_mode="messages" 옵션을 활성화하여 실시간 토큰 및 툴 실행 상태(chunk)를 루프로 획득합니다.
async for stream_mode, chunk in stream_agent.astream(
    {"messages": [{"role": "user", "content": question}], "sources": [], "web_sources": []},
    config,
    stream_mode=cast(Any, ["messages"])
):
    token, metadata = chunk
    
    # 툴(도구) 호출 감지 로직
    # - LLM이 툴을 사용하기로 결심하면, chunk의 tool_calls 또는 tool_call_chunks 부분에 관련 데이터가 인입됩니다.
    names = []
    for tc in (getattr(token, "tool_call_chunks", None) or []):
        if tc.get("name"):
            names.append(tc["name"])
    for tc in (getattr(token, "tool_calls", None) or []):
        if tc.get("name"):
            names.append(tc["name"])
            
    # 감지된 툴 이름을 바탕으로 클라이언트에게 백엔드가 RAG 검색 등을 하고 있음을 알리는 "상태 알림 이벤트"를 시뮬레이션 출력합니다.
    for name in names:
        if name not in announced_tools:
            announced_tools.add(name)
            status_event = {"type": "status", "data": TOOL_STATUS.get(name, "tool")}
            print(f"\n[EVENT: status] -> 백엔드가 {status_event['data']} 도구를 실행하고 있습니다...")

    # 필터링 1: RAG 도구 노드 자체가 동작하면서 내뿜는 임베딩 매칭 결과 데이터(도구 응답)는 사용자 화면에 텍스트 토큰으로 바로 뿌릴 필요가 없으므로 무시합니다.
    if isinstance(metadata, dict) and metadata.get("langgraph_node") == "tools":
        continue
        
    # 필터링 2: 도구 호출 선언 자체를 지칭하는 AI의 내부 특수 토큰은 일반 텍스트 답변이 아니므로 출력에서 걸러줍니다.
    if getattr(token, "tool_calls", None):
        continue
        
    # 실제 사용자가 읽어야 할 AI 답변 텍스트 토큰을 추출하여 콘솔에 실시간 출력(Streaming)합니다.
    content = getattr(token, "content", "")
    if content:
        print(content, end="", flush=True)

print("\n\n✅ 스트리밍 및 이벤트 가공 시뮬레이션 완료!")


Deserializing unregistered type __main__.BioAnswer from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'BioAnswer')]


✅ 스트리밍 전용 에이전트 초기화 완료!
🚀 [Streaming Start] 실시간 스트림 이벤트를 감지하고 토큰을 가공합니다...


[EVENT: status] -> 백엔드가 paper_search 도구를 실행하고 있습니다...
하나의 연구는 행성 생성 과정에서 중요한 단계인 **planetesimal formation**(행성체 형성)에 대해 다양한 관점을 제공합니다. 여기서는 관련된 논문 중 하나를 요약하겠습니다.

### 논문 제목: The multifaceted planetesimal formation process
- **저자**: [출처 미제공]
- **arXiv ID**: 1402.1344

#### 요약:
이 논문은 행성체 형성이 어떻게 이루어지는지에 대한 복잡한 과정을 설명합니다. 행성체는 지구형 행성과 가스, 얼음 거대 행성의 고체 중심이 형성되는 중요한 초기 단계로, 이들은 주로 미세한 먼지와 얼음 입자가 뭉쳐서 생성됩니다. 행성체는 성운에서 형성된 물리적 조건을 기록하는 독특한 존재인데, 이들은 소행성, 외행성 천체 및 혜성과 같은 형태로 남아 있습니다.

논문은 행성체 형성이 미세한 먼지와 얼음 입자로 시작하여 급속히 성장을 이루는 과정을 기술합니다. 주요 내용은 다음과 같습니다:
1. **성장 경로**: 성장 과정 중에는 여러 저해 요소들이 존재하는데, 이는 주로 비효율적인 접착, 파편화 및 방사선 드리프트와 관련이 있습니다. 두 가지 유망한 성장 경로가 제시되는데, 하나는 고속 충돌이 일어나면서 작은 집합체가 대형 목표물에 질량을 이전하는 것입니다. 다른 하나는 내부 밀도가 낮아지면서 접착 확률이 증가하는 ‘부풀린 성장’입니다.

2. **입자 크기의 다양성**: 미세한 입자부터 시작하여 지름 1km에서 1000km까지 다양한 크기의 행성체가 형성됩니다. 이 과정에서 고속의 대기 소용돌이가 중요한 역할을 하며, 과밀한 섬유질이 중력적으로 분열되어 구속된 입자 덩어리로 이어집니다.

3. **하이브리드 모델**: 저자들은 행성체 형

## 6단계. 최종 결과 분석 및 출처 연동 확인

스트리밍이 완료된 직후, 에이전트가 툴(RAG)을 호출하는 과정에서 DB 내 세션 메모리(`sources`, `web_sources`)에 차곡차곡 누적 적재해 둔 데이터들을 일괄 조회합니다.

### 💡 주요 핵심 개념
1. **출처 수집(Citation Accumulation)의 이점**
   - 텍스트 답변 내용이 아무리 빠르게 스트리밍되더라도, 답변의 토대가 된 논문 정보와 링크들은 백엔드 DB 세션에 완벽하게 정형화된 리스트로 보관됩니다.
2. **중복 필터링**
   - LLM이 답변 작성 중 여러 번 툴을 호출하거나 동일 논문을 중복 인용하더라도, 데이터가 겹쳐서 UI에 노출되지 않도록 `arxiv_id` 및 `url` 기준으로 고유한 세트(Set)를 생성해 필터링하는 로직이 적용되어야 합니다.


In [18]:
# 1. 직전 스트리밍 세션의 상태 데이터를 체크포인터로부터 복원해 조회합니다.
unique_sources = []
unique_web = []

if stream_agent is not None:
    try:
        # thread_id 세션에 저장된 최종 상태(state)를 긁어옵니다.
        state = await stream_agent.aget_state(config)
        
        # 2. 누적된 논문 출처 가공 및 중복 제거
        #    - sources 내의 각 논문 데이터를 순회하면서 arxiv_id 기준으로 중복을 제외하고 unique_sources에 담아줍니다.
        raw_sources = state.values.get("sources", []) if state.values else []
        seen_arxiv = set()
        for s in raw_sources:
            if s.get("arxiv_id") and s["arxiv_id"] not in seen_arxiv:
                seen_arxiv.add(s["arxiv_id"])
                unique_sources.append(s)

        # 3. 누적된 일반 웹 사이트(Tavily 검색 등) 소스 가공 및 중복 제거
        #    - url 기준으로 중복을 걸러내어 unique_web 리스트를 만듭니다.
        raw_web = state.values.get("web_sources", []) if state.values else []
        seen_url = set()
        for s in raw_web:
            if s.get("url") and s["url"] not in seen_url:
                seen_url.add(s["url"])
                unique_web.append(s)
    except Exception as e:
        print(f"⚠️ 상태 조회 중 오류 발생 ({e}). 모의 데이터로 시뮬레이션 결과를 출력합니다.")

# 4. (로컬 테스트 시나리오 방어 코드) DB가 연동되지 않았을 시나리오를 대비한 모의 데이터를 정의합니다.
if not unique_sources:
    unique_sources = [
        {
            "arxiv_id": "1402.1344",
            "title": "The multifaceted planetesimal formation process",
            "summary": "Accumulation of dust and ice particles in protoplanetary disks and the physical processes that drive planetesimal growth."
        },
        {
            "arxiv_id": "2112.15413",
            "title": "Contemporary formation of early solar system planetesimals at two distinct radial locations",
            "summary": "This study analyzes the timeline and radial profiles for planetesimal formation from dust reservoirs."
        }
    ]
if not unique_web:
    unique_web = [
        {
            "title": "NASA - Protoplanetary Disks and Planets",
            "url": "https://www.nasa.gov/mission_pages/spitzer/news/spitzer20081027.html"
        }
    ]

# 5. 가공 완료된 출처 카드를 화면에 출력합니다.
#    - 실제 웹 서비스에서는 이 정형 리스트가 REST API 응답에 실려나가며, 프론트엔드가 이를 바탕으로 하단에 '출처 논문 카드'를 동적으로 그리게 됩니다.
print("=== [최종 가공된 논문 출처 목록 (Citations)] ===")
for idx, source in enumerate(unique_sources, 1):
    print(f"{idx}. [{source['arxiv_id']}] {source['title']}")
    print(f"   요약: {source['summary'][:120]}...\n")

print("=== [최종 가공된 웹 검색 출처 목록 (Web Citations)] ===")
for idx, source in enumerate(unique_web, 1):
    print(f"{idx}. {source['title']}")
    print(f"   URL: {source['url']}")


Deserializing unregistered type __main__.BioAnswer from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'BioAnswer')]


=== [최종 가공된 논문 출처 목록 (Citations)] ===
1. [1402.1344] The multifaceted planetesimal formation process
   요약: Title: The multifaceted planetesimal formation process Abstract: Accumulation of dust and ice particles into planetesima...

2. [2112.15413] Contemporary formation of early solar system planetesimals at two   distinct radial locations
   요약: The formation of planetesimals is expected to occur via particle-gas instabilities that concentrate dust into self-gravi...

3. [1803.00575] Planetesimal formation during protoplanetary disk buildup
   요약: Title: Planetesimal formation during protoplanetary disk buildup Abstract: Models of dust coagulation and subsequent pla...

4. [1511.07762] A panoptic model for planetesimal formation and pebble delivery
   요약: The journey from dust particle to planetesimal involves physical processes acting on scales ranging from micrometers (th...

5. [1609.01785] Late veneer and late accretion to the terrestrial planets
   요약: It is generally accepted t